In [ ]:
import json
import shutil
import subprocess
from datetime import datetime
from pathlib import Path
 
import numpy as np
import polars as pl
from Bio import SeqIO

In [ ]:
CWD = Path().resolve()
BASE = CWD.parent

print(f"Current working directory: {CWD}")
print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups_map = {
    "plants_plt": ["solanaceae"],
}
 
RUN_NAME = "first_pass"  # output goes to <subgroup>/cds_tree/mafft/<RUN_NAME>/
MODE = "nt"  # "nt": plain nucleotide alignment ("codon" will be added later)
MAFFT = "mafft"
MAFFT_ARGS = ["--auto", "--thread", "-36"]  # MAFFT picks the strategy
OVERWRITE = True  # True re-aligns every gene and allows changing the settings of an existing run
 
if MODE != "nt":
    raise NotImplementedError(f"MODE={MODE!r} is not implemented yet; use 'nt'")

# thresholds for flagging, not for removing anything
GAPPY_COL = 0.5  # a column is 'gappy' if more than this fraction of sequences has a gap there
LOW_IDENTITY = 0.9  # flag sequences whose identity to the column consensus is below this

if shutil.which(MAFFT) is None:
    raise FileNotFoundError(f"'{MAFFT}' not found in PATH")
mafft_version = subprocess.run([MAFFT, "--version"], capture_output=True, text=True).stderr.strip()
print(f"MAFFT: {mafft_version}")

In [ ]:
def read_fasta(path: Path) -> dict[str, str]:
    return {rec.id: str(rec.seq).upper() for rec in SeqIO.parse(path, "fasta")}
 
 
def check_run_settings(run_json: Path, settings: dict) -> None:
    """Stop if RUN_NAME already exists with different settings (unless OVERWRITE)."""
    if not run_json.exists():
        return
    old = json.loads(run_json.read_text())
    changed = {k: (old.get(k), v) for k, v in settings.items() if old.get(k) != v}
    if changed and not OVERWRITE:
        details = "; ".join(f"{k}: {a!r} -> {b!r}" for k, (a, b) in changed.items())
        raise RuntimeError(
            f"{run_json.parent.name} already exists with different settings ({details}). "
            "Use a new RUN_NAME, or set OVERWRITE = True."
        )
 
 
def run_mafft(fasta_in: Path, aln_out: Path, log_file: Path) -> None:
    """Align one gene; write to a temporary file first so a crash never leaves a partial alignment."""
    cmd = [MAFFT, *MAFFT_ARGS, str(fasta_in)]
    tmp = aln_out.with_suffix(".tmp")
    with open(tmp, "w") as out, open(log_file, "a") as log:
        log.write(f"\n# {datetime.now():%Y-%m-%d %H:%M:%S} | {' '.join(cmd)} > {aln_out.name}\n")
        log.flush()
        result = subprocess.run(cmd, stdout=out, stderr=log)
    if result.returncode != 0:
        tmp.unlink(missing_ok=True)
        raise RuntimeError(f"MAFFT failed on {fasta_in.name} (exit {result.returncode}), see {log_file.name}")
    tmp.replace(aln_out)
 
 
def check_alignment(label: str, unaligned: dict[str, str], aligned: dict[str, str]) -> tuple[dict, list[dict]]:
    """Per-gene summary and per-sequence rows for one alignment."""
    ids = list(aligned)
    mat = np.array([list(aligned[i]) for i in ids], dtype="U1")
    n_seq, aln_len = mat.shape
    gaps = mat == "-"
 
    # consensus = most common non-gap base per column; identity counted on the sequence's own non-gap sites
    consensus = []
    for col in mat.T:
        bases, counts = np.unique(col[col != "-"], return_counts=True)
        consensus.append(bases[counts.argmax()] if len(bases) else "-")
    consensus = np.array(consensus, dtype="U1")
    matches = (mat == consensus) & ~gaps
    non_gap = (~gaps).sum(axis=1)
    identity = np.where(non_gap > 0, matches.sum(axis=1) / np.maximum(non_gap, 1), 0.0)
 
    gap_frac_seq = gaps.mean(axis=1)
    max_in_len = max(len(s) for s in unaligned.values())
 
    gene_row = {
        "label": label,
        "n_seqs_in": len(unaligned),
        "n_seqs_aln": n_seq,
        "same_ids": set(unaligned) == set(aligned),
        "same_sequences": all(aligned[i].replace("-", "") == unaligned.get(i) for i in ids),
        "max_unaligned_len": max_in_len,
        "aln_len": aln_len,
        "inflation": round(aln_len / max_in_len, 3),
        "gap_pct": round(100 * gaps.mean(), 2),
        "gappy_cols": int((gaps.mean(axis=0) > GAPPY_COL).sum()),
        "min_identity": round(float(identity.min()), 3),
        "min_identity_AN": ids[int(identity.argmin())],
        "n_low_identity": int((identity < LOW_IDENTITY).sum()),
    }
    seq_rows = [
        {
            "label": label,
            "AN": an,
            "ungapped_len": int(non_gap[k]),
            "gap_frac": round(float(gap_frac_seq[k]), 3),
            "identity_to_consensus": round(float(identity[k]), 3),
        }
        for k, an in enumerate(ids)
    ]
    return gene_row, seq_rows

In [ ]:
for g, subgs in groups_map.items():
    for subg in subgs:
        subg_dir = BASE / g / subg
        tree_dir = subg_dir / "cds_tree"
        cds_dir = tree_dir / "cds"
        fasta_dir = cds_dir / "fasta"
        qc_file = cds_dir / "label_qc.tsv"
        run_dir = tree_dir / "mafft" / RUN_NAME
        aln_dir = run_dir / "aln"
        aln_dir.mkdir(parents=True, exist_ok=True)
        log_file = run_dir / "mafft.log"
        run_json = run_dir / "run.json"
        print(f"Processing {g}/{subg} | run: {RUN_NAME}")
 
        # --- the settings that define this run; an existing run with other settings is protected
        settings = {"mode": MODE, "mafft_args": MAFFT_ARGS}
        check_run_settings(run_json, settings)
 
        # --- genes and expected sequences come from the QC table, not from the folder contents
        qc = pl.read_csv(qc_file, separator="\t", infer_schema_length=0)
        kept = qc.filter((pl.col("label_included") == "true") & (pl.col("status") == "kept"))
        expected = {lab: set(df["AN"]) for (lab,), df in kept.group_by("label")}
        labels = sorted(expected)
        print(f"  {len(labels)} labels included in {qc_file.relative_to(subg_dir)}")
 
        # --- every input FASTA must exist and match the QC exactly
        unaligned, input_problems = {}, []
        for label in labels:
            fasta = fasta_dir / f"{label}.fasta"
            if not fasta.exists():
                input_problems.append(f"{label}: missing {fasta.name}")
                continue
            seqs = read_fasta(fasta)
            missing, extra = expected[label] - set(seqs), set(seqs) - expected[label]
            if missing or extra:
                input_problems.append(f"{label}: FASTA does not match QC ({len(missing)} missing, {len(extra)} extra)")
                continue
            unaligned[label] = seqs
        if input_problems:
            for p in input_problems:
                print(f"    {p}")
            raise RuntimeError("Input FASTAs do not match the QC table; re-run 02_cds_fasta first")
 
        # --- run.json is written before aligning (status 'running') and completed at the end,
        # so an interrupted run is still recorded with its settings
        run_info = {
            "run_name": RUN_NAME,
            "group": g,
            "subgroup": subg,
            "status": "running",
            "date_started": f"{datetime.now():%Y-%m-%d %H:%M:%S}",
            "date_finished": None,
            **settings,
            "mafft": MAFFT,
            "mafft_path": shutil.which(MAFFT),
            "mafft_version": mafft_version,
            "input_qc": str(qc_file.relative_to(subg_dir)),
            "input_qc_abs": str(qc_file.resolve()),
            "input_fasta_dir": str(fasta_dir.relative_to(subg_dir)),
            "input_fasta_dir_abs": str(fasta_dir.resolve()),
            "n_genes": len(labels),
            "genes": labels,
        }
        run_json.write_text(json.dumps(run_info, indent=2))
 
        # --- align
        n_run = n_skip = 0
        for label in labels:
            fasta, aln = fasta_dir / f"{label}.fasta", aln_dir / f"{label}.fasta"
            if not OVERWRITE and aln.exists() and aln.stat().st_mtime >= fasta.stat().st_mtime:
                n_skip += 1
                continue
            run_mafft(fasta, aln, log_file)
            n_run += 1
        print(f"  aligned {n_run}, skipped {n_skip} (up to date) -> {aln_dir.relative_to(subg_dir)}/")
 
        # --- check every alignment
        gene_rows, seq_rows = [], []
        for label in labels:
            gene_row, rows = check_alignment(label, unaligned[label], read_fasta(aln_dir / f"{label}.fasta"))
            gene_rows.append(gene_row)
            seq_rows.extend(rows)
        aln_qc = pl.DataFrame(gene_rows)
        seq_qc = pl.DataFrame(seq_rows)
        aln_qc.write_csv(run_dir / "aln_qc.tsv", separator="\t")
        seq_qc.write_csv(run_dir / "aln_seq_qc.tsv", separator="\t")
 
        run_info.update(
            status="complete",
            date_finished=f"{datetime.now():%Y-%m-%d %H:%M:%S}",
            total_aln_len=int(aln_qc["aln_len"].sum()),
        )
        run_json.write_text(json.dumps(run_info, indent=2))
 
        # --- summary
        broken = aln_qc.filter(~pl.col("same_ids") | ~pl.col("same_sequences"))
        print(f"  QC -> {run_dir.relative_to(subg_dir)}/aln_qc.tsv (per gene), aln_seq_qc.tsv (per sequence), run.json")
        print(f"  alignments that lost or changed sequences: {broken.height}")
        print(
            f"  total alignment length: {aln_qc['aln_len'].sum():,} bp"
            f" | median gap %: {aln_qc['gap_pct'].median()}"
            f" | median inflation: {aln_qc['inflation'].median()}"
        )
        print("  genes sorted by gap % (top 10):")
        print(
            aln_qc.sort("gap_pct", descending=True)
            .select("label", "n_seqs_aln", "max_unaligned_len", "aln_len", "inflation", "gap_pct", "gappy_cols", "min_identity", "min_identity_AN")
            .head(10)
        )
        low = seq_qc.filter(pl.col("identity_to_consensus") < LOW_IDENTITY).sort("identity_to_consensus")
        print(f"  sequences with identity to consensus < {LOW_IDENTITY}: {low.height}")
        if low.height:
            print(low.group_by("AN").agg(pl.len().alias("genes"), pl.col("label").str.join(",")).sort("genes", descending=True))
 
        stale = {p.stem for p in aln_dir.glob("*.fasta")} - set(labels)
        if stale:
            print(f"  note: {len(stale)} alignments in {aln_dir.relative_to(subg_dir)}/ are from labels no longer included: {sorted(stale)}")

In [ ]:
# Per-sequence view for one gene, worst first
seq_qc.filter(pl.col("label") == "PLT-YCF1").sort("identity_to_consensus")